# Can we predict a warranty claim within a year?
Target: `claim_within_year` (12.6% positive). Features known at sale time: `channel`, `device_age_years`, `price_eur`, `daily_hours` (all recorded at sale/registration per columns.csv). `device_serial` is an identifier and is dropped.

**Split design.** The data has no date column, so there is no 'later rows' to hold out, and every `device_serial` appears exactly once (checked below), so there are no entities to group. A random stratified split is therefore the appropriate one. Row order is not assumed to be time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
df=pd.read_csv('data.csv')
print(df.shape, 'duplicate serials:', df.device_serial.duplicated().sum(), 'missing:', df.isna().sum().sum())
df.describe(include='all')

## 1. Set aside the test set first (20%, stratified), before any exploration or modelling

In [ ]:
FEATURES=['channel','device_age_years','price_eur','daily_hours']; TARGET='claim_within_year'
train,test=train_test_split(df,test_size=0.2,random_state=0,stratify=df[TARGET])
X_tr,y_tr=train[FEATURES],train[TARGET]; X_te,y_te=test[FEATURES],test[TARGET]
print(len(train),len(test),y_tr.mean(),y_te.mean())

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: logistic regression on one-hot channel + standardised numerics.
3. `hgb_shallow`, `hgb_deeper`: gradient boosting (depth 2 / lr 0.05, and depth 4 / lr 0.05) with early stopping.

Selection metric: CV average precision (PR-AUC), given the 12.6% positive rate; ROC-AUC reported too. The decision threshold is chosen to maximise F1 on out-of-fold predictions. All preprocessing lives inside the Pipeline, so it is refit in each fold.

In [ ]:
num=['device_age_years','price_eur','daily_hours']
def prep(scale): 
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['channel']),('n',StandardScaler() if scale else 'passthrough',num)])
def lr(C): return Pipeline([('p',prep(True)),('m',LogisticRegression(C=C,max_iter=1000))])
def hgb(d,lr_): return Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=lr_,early_stopping=True,random_state=0))])
models={'baseline':Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':lr(0.1),'logreg_C1':lr(1),'logreg_C10':lr(10),
 'hgb_shallow':hgb(2,0.05),'hgb_deeper':hgb(4,0.05)}

## 3. Cross-validation on training rows only (5-fold stratified; entities are unique so whole-entity folds hold automatically)

In [ ]:
cv=StratifiedKFold(5,shuffle=True,random_state=1)
oof={};rows=[]
for n,m in models.items():
    p=cross_val_predict(m,X_tr,y_tr,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    rows.append((n,roc_auc_score(y_tr,p),average_precision_score(y_tr,p)))
cvres=pd.DataFrame(rows,columns=['model','cv_roc_auc','cv_pr_auc']).set_index('model').sort_values('cv_pr_auc',ascending=False)
cvres

In [ ]:
best=cvres.index[0]
ths=np.linspace(0.02,0.6,59)
f1s=[f1_score(y_tr,oof[best]>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'threshold:',round(thr,3),'CV F1:',round(max(f1s),3))
print('CV precision/recall at threshold:',precision_score(y_tr,oof[best]>=thr),recall_score(y_tr,oof[best]>=thr))

## 4. Decision (written before touching the test set)
Model = the top CV PR-AUC model above, with the F1-maximising out-of-fold threshold printed above. Features: the four listed. No further tuning after this point.

## 5. Refit on all training rows; score the test set once

In [ ]:
final=models[best].fit(X_tr,y_tr)
pt=final.predict_proba(X_te)[:,1]
base=models['baseline'].fit(X_tr,y_tr).predict_proba(X_te)[:,1]
print(f'TEST {best}: ROC-AUC={roc_auc_score(y_te,pt):.3f}  PR-AUC={average_precision_score(y_te,pt):.3f}  (prevalence {y_te.mean():.3f})')
print(f'  at thr={thr:.3f}: precision={precision_score(y_te,pt>=thr):.3f} recall={recall_score(y_te,pt>=thr):.3f} F1={f1_score(y_te,pt>=thr):.3f}')
print(f'TEST baseline: ROC-AUC={roc_auc_score(y_te,base):.3f} PR-AUC={average_precision_score(y_te,base):.3f}')

## Conclusion
See the printed test-set headline above; interpret relative to the baseline and the 12.6% prevalence. Caveats: no time information, so generalisation to future sales is untested; the test set (~1,400 rows, ~176 positives) gives a fairly wide uncertainty on these metrics.